# Setup

In [1]:
!pip install -q llama-index-llms-anthropic llama-index-tools-code-interpreter llama-index

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.4/243.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.9/253.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 7.9 MB/s eta 0:00:00


Instalowane są następujące pakiety:

1. `llama-index-llms-anthropic` - to biblioteka integrująca modele językowe Anthropic z frameworkiem LlamaIndex. Pozwala na wykorzystanie modeli AI od Anthropic (takich jak Claude) w aplikacjach opartych na LlamaIndex.

2. `llama-index-tools-code-interpreter` - narzędzie z ekosystemu LlamaIndex służące do interpretacji i wykonywania kodu. Umożliwia modelom AI uruchamianie kodu w czasie rzeczywistym.

3. `llama-index` - główna biblioteka LlamaIndex, która służy do budowania aplikacji wykorzystujących modele językowe z dostępem do danych zewnętrznych. Pomaga w indeksowaniu, wyszukiwaniu i przetwarzaniu danych dla sztucznej inteligencji.

In [ ]:
from google.colab import userdata
import os
from llama_index.tools.code_interpreter.base import CodeInterpreterToolSpec
from llama_index.llms.anthropic import Anthropic
from llama_index.core import Settings
from llama_index.core.agent import FunctionCallingAgent

`from llama_index.tools.code_interpreter.base import CodeInterpreterToolSpec` - importuje specyfikację narzędzia do interpretacji kodu z pakietu LlamaIndex, co umożliwi wykonywanie kodu w czasie rzeczywistym.

`from llama_index.llms.anthropic import Anthropic` - importuje klasę do wykorzystania modeli językowych Anthropic (jak Claude) w projekcie.

`from llama_index.core import Settings` - importuje ustawienia globalne dla LlamaIndex.

`from llama_index.core.agent import FunctionCallingAgent` - importuje agenta, który może wywoływać funkcje na podstawie poleceń w języku naturalnym.

In [ ]:
class CFG:
    model = "claude-3-5-sonnet-20241022"

In [ ]:
os.environ["ANTHROPIC_API_KEY"] = userdata.get("claude")

# Agent

In [5]:
code_spec = CodeInterpreterToolSpec()

tools = code_spec.to_tool_list()

Te dwie linie kodu ustawiają narzędzie do interpretacji kodu, które będzie używane przez model językowy.

`code_spec = CodeInterpreterToolSpec()` tworzy nową instancję specyfikacji interpretera kodu. Jest to obiekt z biblioteki LlamaIndex, który definiuje jak kod Python powinien być wykonywany w środowisku agenta AI.

`tools = code_spec.to_tool_list()` konwertuje tę specyfikację na listę narzędzi. Taki format jest wymagany przez agenta, który będzie używał tych narzędzi do wykonywania operacji. Metoda `to_tool_list()` przekształca specyfikację w format, który agent może zrozumieć i wykorzystać.

Te linie przygotowują środowisko, w którym model językowy będzie mógł wykonywać kod Python na żądanie - na przykład do analizy danych finansowych z Yahoo Finance (zaimportowanego wcześniej jako `yf`), przetwarzania danych w pandas, czy operacji na datach.

In [ ]:
tokenizer = Anthropic().tokenizer
Settings.tokenizer = tokenizer

llm_claude = Anthropic(model=CFG.model)


agent = FunctionCallingAgent.from_tools(
    tools,
    llm=llm_claude,
    verbose=True,
    allow_parallel_tool_calls=False,
)

Te linie kodu konfigurują model językowy Anthropic (Claude) i tworzą agenta, który będzie wykorzystywał ten model wraz z przygotowanymi wcześniej narzędziami.

`tokenizer = Anthropic().tokenizer` pobiera tokenizer z instancji klasy Anthropic. Tokenizer to komponent, który dzieli tekst na mniejsze jednostki (tokeny), co jest niezbędnym krokiem w przetwarzaniu tekstu przez modele językowe. Tokenizer Claude'a ma specyficzny sposób podziału tekstu, który jest dopasowany do architektury modelu.

`Settings.tokenizer = tokenizer` ustawia pobrany tokenizer jako globalny tokenizer w ustawieniach LlamaIndex. Dzięki temu cały framework będzie używał tego samego tokenizera, co zapewni spójność w liczeniu i przetwarzaniu tokenów.

`llm_claude = Anthropic(model = CFG.model)` tworzy instancję modelu Claude, używając identyfikatora modelu zdefiniowanego wcześniej w klasie CFG. Ta instancja będzie służyła do generowania odpowiedzi i przetwarzania zapytań.

`agent = FunctionCallingAgent.from_tools(...)` tworzy agenta, który łączy model językowy z narzędziami. Parametry tej funkcji są następujące:

- `tools` - lista narzędzi (w tym przypadku nasz interpreter kodu), które agent będzie mógł wykorzystywać
- `llm=llm_claude` - wskazuje, że agent ma korzystać z modelu Claude skonfigurowanego wcześniej
- `verbose=True` - włącza tryb szczegółowy, co oznacza, że agent będzie wyświetlał więcej informacji o swoich działaniach (przydatne do debugowania)
- `allow_parallel_tool_calls=False` - wyłącza możliwość równoległego wywoływania narzędzi; narzędzia będą wywoływane sekwencyjnie

In [ ]:
stock = "TESLA"

prompt = f"""
Write a python code to :
- Detect which date is today
- Based on this date, fetch historical prices of {stock} from the beginning of the month until today.
- Analyze last month prices
"""

resp = agent.chat(prompt)

> Running step 56221a3b-e849-4c95-88c0-66267ae9ae58. Step input: 
Write a python code to :
- Detect which date is today
- Based on this date, fetch historical prices of TESLA from the beginning of the month until today.
- Analyze last month prices

Added user message to memory: 
Write a python code to :
- Detect which date is today
- Based on this date, fetch historical prices of TESLA from the beginning of the month until today.
- Analyze last month prices

=== LLM Response ===
I'll help you write a Python code to analyze Tesla's stock prices. We'll use the `yfinance` library for fetching stock data and `pandas` for analysis. Let me break this down into steps:
=== Calling Function ===
Calling function: code_interpreter with args: {}
=== Function Output ===
Encountered error: CodeInterpreterToolSpec.code_interpreter() missing 1 required positional argument: 'code'
> Running step b141db1b-cca8-41ba-af2e-2239f48ba5e4. Step input: None
=== LLM Response ===
Let me write and execute the cod

Ten kod tworzy instancję agenta Claude i używa go do generowania i analizowania kodu Pythona dotyczącego cen akcji Tesli.

`stock = 'TESLA'` definiuje zmienną przechowującą nazwę spółki, której dane akcji będą analizowane - w tym przypadku jest to Tesla.

`prompt = f"""..."""` tworzy zapytanie tekstowe używając f-stringa (formatowanego ciągu znaków), które będzie przesłane do agenta. Zapytanie zawiera instrukcje do:
- Wykrycia bieżącej daty
- Pobrania historycznych cen akcji Tesli od początku miesiąca do dnia dzisiejszego
- Analizy cen z ostatniego miesiąca

`resp = agent.chat(prompt)` wywołuje metodę chat agenta, przekazując mu przygotowane zapytanie. Agent przetworzy to zapytanie, wykona niezbędne operacje (w tym uruchomi kod Python za pomocą narzędzia interpretacji kodu) i przygotuje odpowiedź.

Agent wykorzysta model Claude 3.5 Sonnet (określony wcześniej w konfiguracji), aby zrozumieć zapytanie, a następnie:
1. Napisze kod wykrywający bieżącą datę (prawdopodobnie używając modułu datetime)
2. Użyje biblioteki yfinance do pobrania danych historycznych cen akcji Tesli
3. Przeprowadzi analizę tych cen
4. Przedstawi wyniki w sposób zrozumiały dla człowieka

Wynik tego działania zostanie zapisany w zmiennej `resp`, która będzie zawierać odpowiedź agenta wraz z analizą cen akcji Tesli. Ta odpowiedź może zawierać zarówno kod źródłowy, jak i jego wyniki, a także interpretację tych wyników w ludzkim języku.

In [8]:
print(resp.response)

I've written and executed a code that:

1. Gets today's date (showing as April 3, 2025 in the test environment)
2. Fetches Tesla (TSLA) stock data for:
   - Current month (April 1, 2025 to April 3, 2025)
   - Last month (March 2025)
3. Provides analysis of the data:

For the current month (April 2025 so far):
- Mean price: $275.61
- Minimum price: $268.46
- Maximum price: $282.76

For last month (March 2025):
- Average price: $255.64
- Highest price: $288.14
- Lowest price: $222.15
- Price volatility (standard deviation): $20.64
- Overall price change: -7.41%

The analysis shows that Tesla stock had significant volatility in March 2025, with a price range of about $66 (difference between highest and lowest). The stock ended March with a negative return of -7.41%.

Note: There's a deprecation warning in the code which doesn't affect the results but suggests using `.iloc` for positional indexing in future versions of pandas. Would you like me to modify the code to remove this warning or 

Access to the Agent Memory

In [9]:
agent.memory

ChatMemoryBuffer(chat_store=SimpleChatStore(store={'chat_history': [ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='\nWrite a python code to :\n- Detect which date is today\n- Based on this date, fetch historical prices of TESLA from the beginning of the month until today.\n- Analyze last month prices\n')]), ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'tool_calls': [{'id': 'toolu_01UPUGCebVSsVXsEVmS1rjGf', 'input': {}, 'name': 'code_interpreter', 'type': 'tool_use'}], 'thinking': None}, blocks=[TextBlock(block_type='text', text="I'll help you write a Python code to analyze Tesla's stock prices. We'll use the `yfinance` library for fetching stock data and `pandas` for analysis. Let me break this down into steps:")]), ChatMessage(role=<MessageRole.TOOL: 'tool'>, additional_kwargs={'name': 'code_interpreter', 'tool_call_id': 'toolu_01UPUGCebVSsVXsEVmS1rjGf'}, blocks=[TextBlock(block_type='text', te